# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset with the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, following best practices for referencing dataset entities by their `@id` fields.

### Dataset Source
This dataset source is described by a Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
md = dataset.metadata

# Print dataset name and description
print(f"{md.name}: {md.description}\n")

## 2. Data Overview
Let's discover the available record sets, their IDs, fields, and columns present in the dataset. All identifiers used are referenced by their `@id` as per the Croissant standard.

In [ ]:
# List all record sets by @id
if not hasattr(md, 'record_sets'):
    # Fallback for older versions or special property names
    record_sets = getattr(md, 'recordSet', [])
else:
    record_sets = md.record_sets

print("Record sets available in this dataset:")
record_sets_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    print(f"  - {rs_id}")
    record_sets_ids.append(rs_id)
# If no record sets listed in metadata, try extracting from the Dataset object's _metadata
if not record_sets_ids and hasattr(md, '_metadata'):
    # Internal workaround for croissant datasets that don't list recordSets directly
    croissant = md._metadata
    if isinstance(croissant, dict) and 'recordSet' in croissant:
        record_sets = croissant['recordSet']
        for rs in record_sets:
            rs_id = rs['@id']
            print(f"  - {rs_id}")
            record_sets_ids.append(rs_id)

if not record_sets_ids:
    print("[!] No record sets found in schema. Dataset may not contain tabular resources.")
else:
    # For each record set, print its fields/columns by @id
    print("\nFields/columns by record set:")
    # We'll use the first record set as an exploration example
    for rs in record_sets:
        rs_obj = rs if isinstance(rs, dict) else {'@id': rs}
        rs_id = rs_obj['@id']
        print(f"\nRecord set: {rs_id}")
        if 'field' in rs_obj:
            fields = rs_obj['field']
        elif 'fields' in rs_obj:
            fields = rs_obj['fields']
        else:
            fields = []
        if fields:
            for f in fields:
                f_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
                print(f"  - Field: {f_id}")
        elif 'column' in rs_obj:
            cols = rs_obj['column']
            for c in cols:
                c_id = c['@id'] if isinstance(c, dict) and '@id' in c else c
                print(f"  - Column: {c_id}")
        else:
            print("  [!] No fields or columns listed.")

## 3. Data Extraction
Using the identified `@id` of each record set, we'll load records from all sets into DataFrames for analysis. We'll show an example preview for the first available record set.


In [ ]:
# Load data from all record sets by @id
dataframes = {}

for rs_id in record_sets_ids:
    try:
        print(f"Loading data for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print(f"  No records found for {rs_id}.")
    except Exception as e:
        print(f"  Failed to load {rs_id} ({e})")

# For exploration, pick the first successfully loaded record set
if dataframes:
    main_rs_id = list(dataframes)[0]
    print(f"\nPreview of first record set ({main_rs_id}):")
    display_cols = dataframes[main_rs_id].columns.tolist()
    print(display_cols)
    display(dataframes[main_rs_id].head())
else:
    main_rs_id = None
    print("No tabular data extracted.")

## 4. Exploratory Data Analysis (EDA)
Let's conduct some basic analysis using the `@id`-referenced fields and columns. We'll apply:
- Filtering numeric fields
- Normalizing values
- Grouping by a categorical field (if available)


In [ ]:
import numpy as np

# This step depends on dataset details; we auto-select a numeric field if present
if main_rs_id is not None:
    df = dataframes[main_rs_id]
    
    # Find numeric columns (float or int)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        # Sometimes numeric fields are stored as string - try to infer
        possible_num_cols = [c for c in df.columns if df[c].dtype == object]
        col_stats = {}
        for c in possible_num_cols:
            try:
                _asnum = pd.to_numeric(df[c])
                # Accept col if 80%+ non-nulls parse as number
                col_stats[c] = (_asnum.notnull().mean() > 0.8)
            except Exception:
                col_stats[c] = False
        likely_numeric = [c for c, ok in col_stats.items() if ok]
        if likely_numeric:
            # We'll coerce first likely numeric
            numeric_field = likely_numeric[0]
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        else:
            numeric_field = None
    else:
        numeric_field = num_cols[0]
    
    if numeric_field:
        print(f"Selected field for numeric analysis: {numeric_field}")
        threshold = df[numeric_field].mean()  # Use the mean as filter threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered to {len(filtered_df)} records with {numeric_field} > {threshold:.2f}.")
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

        # Try grouping by a non-numeric field
        candidate_categories = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
        group_field = candidate_categories[0] if candidate_categories else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean of {numeric_field} by {group_field} (top 5):")
            print(grouped_df.head())
    else:
        print("No numeric field found suitable for EDA.")
else:
    print("No data frame to analyze.")

## 5. Visualization
Here we visualize the distribution of the selected numeric field and relationships to the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in {main_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field is not None:
        # Plot grouped means (show only top 10 for readability)
        means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,5))
        sns.barplot(x=means.index, y=means.values)
        plt.title(f"Mean {numeric_field} by {group_field} (top 10)")
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()
else:
    print("Insufficient numeric data for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR² dataset on extracellular vesicles using Croissant metadata and the `mlcroissant` library. We:
- Loaded and inspected the dataset schema and contents using `@id` references.
- Extracted tabular records, performed basic filtering, normalization and grouping.
- Visualized key numeric features and their relationships.

This EDA provides a framework for further scientific analysis or tailored transformation of the dataset.